# IPL Analytics - Phase 2

This notebook handles data cleaning, calculates key metrics, and exports professional visualizations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Step 1: Load and Clean Data

In [ ]:
print("🔄 Loading dataset...")
df = pd.read_csv('ipl_matches.csv', low_memory=False)

# Drop matches with no result
df = df.dropna(subset=['winner', 'toss_winner'])

# Convert season to numeric
df['season'] = pd.to_numeric(df['season'], errors='coerce')

# Team name cleanup
team_cleanup = {
    'Delhi Daredevils': 'Delhi Capitals',
    'Kings XI Punjab': 'Punjab Kings',
    'Deccan Chargers': 'Sunrisers Hyderabad',
    'Pune Warriors': 'Rising Pune Supergiant',
    'Rising Pune Supergiants': 'Rising Pune Supergiant'
}
for col in ['team1', 'team2', 'toss_winner', 'winner', 'batting_team']:
    if col in df.columns:
        df[col] = df[col].replace(team_cleanup)

print("✅ Data cleaned!")

## Problem 1: Toss Win Rate vs Toss Lose Rate

In [ ]:
# Match-level analysis
match_level = df.groupby('match_id').first().reset_index()
match_level['toss_winner_won'] = match_level['toss_winner'] == match_level['winner']
toss_win_pct = match_level['toss_winner_won'].mean() * 100
toss_loss_pct = 100 - toss_win_pct

print(f"\n📊 Toss Win Rate: {toss_win_pct:.2f}%")
print(f"📊 Toss Loser Win Rate: {toss_loss_pct:.2f}%")

In [ ]:
# Chart 1: Toss Impact
plt.figure(figsize=(7, 5))
sns.set_style("whitegrid")
bars = plt.bar(['Toss Winners', 'Toss Losers'], [toss_win_pct, toss_loss_pct], 
               color=['#1f77b4', '#aec7e8'], edgecolor='black', width=0.4)

plt.title('IPL Win Rate: Toss Winners vs Toss Losers', fontsize=14, fontweight='bold', pad=15)
plt.ylabel('Match Win Percentage (%)', fontsize=12)
plt.ylim(0, 100)

for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 2, f'{height:.2f}%', 
             ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('chart1_toss_impact.png', dpi=300)
plt.show()
print("💾 Saved: chart1_toss_impact.png")

## Problem 2: Match Phase Impact Analysis

In [ ]:
# Assign cricket phases
def assign_cricket_phase(over):
    if over < 6:
        return 'Powerplay (Overs 1-6)'
    elif over < 15:
        return 'Middle Overs (Overs 7-15)'
    else:
        return 'Death Overs (Overs 16-20)'

df['match_phase'] = df['over'].apply(assign_cricket_phase)
df['is_winning_side'] = df['batting_team'] == df['winner']

# Aggregate runs by phase
phase_runs_agg = df.groupby(['match_id', 'is_winning_side', 'match_phase'])['runs_total'].sum().reset_index()
phase_comparison = phase_runs_agg.groupby(['is_winning_side', 'match_phase'])['runs_total'].mean().unstack(level=0)
phase_comparison.columns = ['Losing Teams', 'Winning Teams']
phase_ordering = ['Powerplay (Overs 1-6)', 'Middle Overs (Overs 7-15)', 'Death Overs (Overs 16-20)']
phase_comparison = phase_comparison.reindex(phase_ordering)

print("\n📈 Average Runs Scored per Phase:")
print(phase_comparison)

In [ ]:
# Chart 2: Phase Runs
plt.figure(figsize=(10, 6))
phase_comparison.plot(kind='bar', color=['#e74c3c', '#2ecc71'], edgecolor='black', width=0.6, ax=plt.gca())
plt.title('Average Runs per Match Phase: Winning vs Losing Teams', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Match Phase', fontsize=12)
plt.ylabel('Average Runs Scored', fontsize=12)
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.legend(title="Match Result")
plt.tight_layout()
plt.savefig('chart2_phase_runs.png', dpi=300)
plt.show()
print("💾 Saved: chart2_phase_runs.png")

## Problem 3: Top Performers (Last 5 Seasons)

In [ ]:
# Get last 5 seasons
available_seasons = sorted(df['season'].dropna().unique())
recent_5_seasons = [int(s) for s in available_seasons[-5:]]
print(f"\n🔍 Analyzing top players across seasons: {recent_5_seasons}")

df_recent = df[df['season'].isin(recent_5_seasons)]

# Top 5 Batters
top_5_batters = df_recent.groupby('batter')['runs_batter'].sum().reset_index()
top_5_batters = top_5_batters.sort_values(by='runs_batter', ascending=False).head(5).reset_index(drop=True)
top_5_batters.columns = ['Player Name (Batter)', 'Total Runs']

# Top 5 Bowlers
valid_dismissals = ['caught', 'bowled', 'lbw', 'stumped', 'caught and bowled', 'hit wicket']
bowler_df = df_recent[df_recent['wicket_kind'].isin(valid_dismissals)]
top_5_bowlers = bowler_df.groupby('bowler').size().reset_index(name='Total Wickets')
top_5_bowlers = top_5_bowlers.sort_values(by='Total Wickets', ascending=False).head(5).reset_index(drop=True)
top_5_bowlers.columns = ['Player Name (Bowler)', 'Total Wickets']

print("\n🏆 Top 5 Batters Leaderboard:")
print(top_5_batters)
print("\n🏆 Top 5 Bowlers Leaderboard:")
print(top_5_bowlers)

In [ ]:
print("\n✅ Analysis Complete!")